# 🎯 Centro de um Grafo

Neste notebook final sobre árvores, estudaremos o conceito de **centro de um grafo** e algoritmos para determiná-lo.

## Objetivos

- Compreender excentricidade de um vértice
- Definir centro e raio de um grafo
- Aprender o método de eliminação progressiva
- Demonstrar que toda árvore tem centro ou bicentro

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from collections import deque

plt.rcParams['figure.figsize'] = (14, 9)
plt.rcParams['font.size'] = 11

## 🌍 Motivação: Escolhendo um Líder

### Problema

Em um grupo de comunicação, queremos escolher um **líder** baseado na **facilidade de acesso** às outras pessoas.

**Critério**: O líder deve estar o mais "próximo" possível de todos os outros membros.

### Modelagem

- **Vértices**: Pessoas
- **Arestas**: Canais de comunicação
- **Distância**: Número de intermediários na comunicação
- **Objetivo**: Minimizar a **distância máxima** do líder para qualquer pessoa

## 📐 Excentricidade de um Vértice

> **Definição**: A **excentricidade** de um vértice $v$, denotada $E(v)$, é a **distância máxima** entre $v$ e todos os outros vértices do grafo $G(V,A)$:
>
> $$E(v) = \max_{w \in V} d(v, w)$$

### Interpretação

- $E(v)$ = "o quão longe $v$ está do vértice mais distante"
- Vértice com **baixa excentricidade** = "central"
- Vértice com **alta excentricidade** = "periférico"

### Propriedades

- $E(v) \geq 0$
- Para árvores: $E(v)$ é bem definido (existe caminho único)

In [ ]:
# Calcular excentricidades em uma árvore
T = nx.Graph()
T.add_edges_from([
    (1, 2), (1, 3),
    (2, 4), (2, 5),
    (3, 6), (3, 7),
    (4, 8), (5, 9)
])

# Calcular excentricidade de cada vértice
excentricidades = nx.eccentricity(T)

# Normalizar cores (vermelho = baixa exc., azul = alta exc.)
max_exc = max(excentricidades.values())
min_exc = min(excentricidades.values())

# Criar mapa de cores
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

norm = Normalize(vmin=min_exc, vmax=max_exc)
cmap = plt.cm.RdYlGn_r  # Vermelho (baixo) → Verde → Amarelo → Vermelho (alto)
cores = [cmap(norm(excentricidades[v])) for v in T.nodes()]

# Visualizar
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(T, seed=42)
nx.draw(T, pos, with_labels=True, node_color=cores,
        node_size=1500, font_size=14, font_weight='bold', 
        edge_color='gray', width=3, cmap=cmap)

# Adicionar rótulos de excentricidade
labels_exc = {v: f"E={excentricidades[v]}" for v in T.nodes()}
pos_labels = {v: (x, y-0.15) for v, (x, y) in pos.items()}
nx.draw_networkx_labels(T, pos_labels, labels_exc, font_size=10, 
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Barra de cor
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=plt.gca(), orientation='vertical', pad=0.01)
cbar.set_label('Excentricidade E(v)', fontsize=12, fontweight='bold')

plt.title('Excentricidades dos Vértices\n(cores: verde=baixa, vermelho=alta)',
          fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

# Tabela de excentricidades
print("\n📊 Excentricidades:")
print("="*50)
print(f"{'Vértice':<10} {'Excentricidade E(v)'}")
print("="*50)
for v in sorted(T.nodes()):
    print(f"{v:<10} {excentricidades[v]}")

print(f"\nMínima excentricidade: {min_exc}")
print(f"Máxima excentricidade: {max_exc}")

## 🎯 Centro de um Grafo

> **Definição**: O **centro** de um grafo $G$ é o **subconjunto de vértices com excentricidade mínima**.
>
> $$C(G) = \{v \in V : E(v) = \min_{w \in V} E(w)\}$$

### Raio do Grafo

> **Definição**: O **raio** de um grafo $G$ é a **mínima excentricidade**:
>
> $$r(G) = \min_{v \in V} E(v)$$

### Diâmetro do Grafo

> **Definição**: O **diâmetro** de um grafo $G$ é a **máxima excentricidade**:
>
> $$d(G) = \max_{v \in V} E(v)$$

### Relações

- $r(G) \leq d(G)$
- $C(G) = \{v : E(v) = r(G)\}$

In [ ]:
# Encontrar centro, raio e diâmetro
def analisar_centro(G):
    """
    Analisa centro, raio e diâmetro de um grafo
    """
    exc = nx.eccentricity(G)
    raio = nx.radius(G)
    diametro = nx.diameter(G)
    centro = nx.center(G)
    periferia = nx.periphery(G)
    
    print(f"\n📊 Análise do Grafo:")
    print("="*60)
    print(f"  Raio r(G): {raio} (mínima excentricidade)")
    print(f"  Diâmetro d(G): {diametro} (máxima excentricidade)")
    print(f"  Centro C(G): {centro}")
    print(f"  Periferia P(G): {periferia}")
    print(f"  |Centro|: {len(centro)}")
    
    return exc, raio, diametro, centro, periferia

# Analisar a árvore anterior
exc, raio, diam, centro, periferia = analisar_centro(T)

# Visualizar destacando centro e periferia
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(T, seed=42)

# Cores: vermelho=centro, azul=periferia, cinza=outros
cores = []
for v in T.nodes():
    if v in centro:
        cores.append('red')
    elif v in periferia:
        cores.append('lightblue')
    else:
        cores.append('lightgray')

nx.draw(T, pos, with_labels=True, node_color=cores,
        node_size=1500, font_size=14, font_weight='bold',
        edge_color='gray', width=3)

# Legenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', label=f'Centro (E={raio})'),
    Patch(facecolor='lightblue', label=f'Periferia (E={diam})'),
    Patch(facecolor='lightgray', label='Outros')
]
plt.legend(handles=legend_elements, loc='upper right', fontsize=11)

plt.title(f'Centro e Periferia da Árvore\nCentro: {centro}, Raio: {raio}, Diâmetro: {diam}',
          fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

## 🌳 Centro em Árvores: Propriedade Especial

> **Proposição**: O centro de uma árvore possui **um ou dois vértices**.
>
> - Se |C(T)| = 1: **centro**
> - Se |C(T)| = 2: **bicentro** (os dois vértices estão conectados)

### Por quê?

Em árvores, existe **caminho único** entre vértices. Logo:
- Se há 3+ vértices no centro, pelo menos um deles não minimiza a distância máxima
- Se |C| = 2, os dois vértices devem ser adjacentes (senão o vértice entre eles seria melhor)

## 🔄 Método de Eliminação Progressiva

> **Algoritmo**: Para determinar o centro de uma árvore $T$:
>
> 1. Elimine **progressivamente** os vértices pendentes (grau 1) e suas arestas
> 2. Repita até restar:
>    - **Um vértice isolado** → centro
>    - **Dois vértices ligados por uma aresta** → bicentro

### Por quê funciona?

> **Proposição**: Seja $T$ uma árvore com pelo menos 3 vértices. A árvore $T'$ obtida pela **exclusão dos vértices pendentes** de $T$ possui o **mesmo centro** que $T$.

### Demonstração (ideia)

- Vértices pendentes nunca estão no centro (têm excentricidade máxima)
- Removê-los reduz em 1 a excentricidade de seus vizinhos
- As excentricidades relativas se mantêm
- Logo: centro se preserva

In [ ]:
def encontrar_centro_eliminacao(T):
    """
    Encontra o centro de uma árvore pelo método de eliminação progressiva.
    Retorna a sequência de árvores e o centro final.
    """
    arvores = [T.copy()]
    T_atual = T.copy()
    
    while T_atual.number_of_nodes() > 2:
        # Encontrar pendentes (grau 1)
        pendentes = [v for v in T_atual.nodes() if T_atual.degree(v) == 1]
        
        if not pendentes:
            break
        
        # Remover pendentes
        T_atual.remove_nodes_from(pendentes)
        arvores.append(T_atual.copy())
    
    # Centro = vértices restantes
    centro = list(T_atual.nodes())
    
    return arvores, centro

# Aplicar método
T_exemplo = nx.Graph()
T_exemplo.add_edges_from([
    (1, 2), (2, 3), (3, 4), (4, 5),  # caminho principal
    (2, 6), (3, 7), (4, 8)           # ramificações
])

arvores_seq, centro_final = encontrar_centro_eliminacao(T_exemplo)

# Visualizar passo a passo
n_passos = len(arvores_seq)
fig, axes = plt.subplots(2, (n_passos+1)//2, figsize=(16, 8))
axes = axes.flatten() if n_passos > 2 else [axes] if n_passos == 1 else axes

for i, T_passo in enumerate(arvores_seq):
    if i < len(axes):
        ax = axes[i]
        
        # Identificar pendentes e outros
        pendentes = [v for v in T_passo.nodes() if T_passo.degree(v) == 1]
        cores = ['lightcoral' if v in pendentes else 'lightgreen' for v in T_passo.nodes()]
        
        pos = nx.spring_layout(T_passo, seed=42)
        nx.draw(T_passo, pos, ax=ax, with_labels=True, node_color=cores,
                node_size=800, font_size=12, font_weight='bold',
                edge_color='gray', width=2)
        
        n = T_passo.number_of_nodes()
        ax.set_title(f'Passo {i+1}\n(n={n}, pendentes em vermelho)', 
                    fontsize=10, fontweight='bold')
        ax.axis('off')

# Remover eixos extras
for i in range(len(arvores_seq), len(axes)):
    axes[i].axis('off')

plt.suptitle(f'Método de Eliminação Progressiva\nCentro encontrado: {centro_final}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n🎯 Centro da árvore: {centro_final}")
if len(centro_final) == 1:
    print("  → CENTRO (1 vértice)")
else:
    print("  → BICENTRO (2 vértices adjacentes)")

## 🔍 Verificação: Centro vs. Eliminação

In [ ]:
# Comparar método de eliminação com cálculo direto
def comparar_metodos(T):
    """
    Compara método de eliminação com cálculo direto de centro
    """
    # Método direto (NetworkX)
    centro_direto = nx.center(T)
    
    # Método de eliminação
    _, centro_eliminacao = encontrar_centro_eliminacao(T)
    
    # Comparar
    iguais = set(centro_direto) == set(centro_eliminacao)
    
    print(f"\n🔍 Comparação de Métodos:")
    print("="*60)
    print(f"  Centro (cálculo direto): {sorted(centro_direto)}")
    print(f"  Centro (eliminação):     {sorted(centro_eliminacao)}")
    print(f"  Iguais? {iguais} {'✓' if iguais else '✗'}")
    
    return iguais

# Testar com várias árvores
print("\n" + "="*60)
print("TESTANDO MÉTODO DE ELIMINAÇÃO")
print("="*60)

# Teste 1: Caminho
print("\nTeste 1: Caminho (7 vértices)")
T1 = nx.path_graph(7)
comparar_metodos(T1)

# Teste 2: Árvore balanceada
print("\nTeste 2: Árvore balanceada")
T2 = nx.balanced_tree(2, 3)
comparar_metodos(T2)

# Teste 3: Árvore aleatória
print("\nTeste 3: Árvore aleatória (10 vértices)")
T3 = nx.random_tree(10, seed=42)
comparar_metodos(T3)

print("\n✓ Método de eliminação SEMPRE encontra o centro correto!")

## 📊 Exemplos: Centro vs. Bicentro

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Árvore com CENTRO (1 vértice)
T_centro = nx.star_graph(5)  # estrela tem centro único
centro1 = nx.center(T_centro)
pos1 = nx.spring_layout(T_centro, seed=42)
cores1 = ['red' if v in centro1 else 'lightblue' for v in T_centro.nodes()]
nx.draw(T_centro, pos1, ax=axes[0], with_labels=True, node_color=cores1,
        node_size=1200, font_size=14, font_weight='bold', edge_color='gray', width=3)
axes[0].set_title(f'CENTRO\nC(T) = {centro1} (1 vértice)',
                 fontsize=12, fontweight='bold')
axes[0].axis('off')

# 2. Árvore com BICENTRO (2 vértices)
T_bicentro = nx.path_graph(6)  # caminho par tem bicentro
centro2 = nx.center(T_bicentro)
pos2 = {i: (i, 0) for i in T_bicentro.nodes()}
cores2 = ['red' if v in centro2 else 'lightgreen' for v in T_bicentro.nodes()]
nx.draw(T_bicentro, pos2, ax=axes[1], with_labels=True, node_color=cores2,
        node_size=1200, font_size=14, font_weight='bold', edge_color='gray', width=3)
axes[1].set_title(f'BICENTRO\nC(T) = {centro2} (2 vértices adjacentes)',
                 fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("\n📌 Observações:")
print("  ✓ Estrela → CENTRO (1 vértice central)")
print("  ✓ Caminho par → BICENTRO (2 vértices centrais adjacentes)")
print("  ✓ Caminho ímpar → CENTRO (1 vértice central)")

## 📐 Propriedade: Preservação do Centro

In [ ]:
# Demonstrar que remover pendentes preserva o centro
T = nx.random_tree(12, seed=100)

# Centro original
centro_original = nx.center(T)

# Remover pendentes uma vez
T_reduzido = T.copy()
pendentes = [v for v in T_reduzido.nodes() if T_reduzido.degree(v) == 1]
T_reduzido.remove_nodes_from(pendentes)

# Centro após remoção
centro_reduzido = nx.center(T_reduzido)

print("\n🔍 Verificando Preservação do Centro:")
print("="*60)
print(f"  Árvore original: {T.number_of_nodes()} vértices")
print(f"  Centro original: {centro_original}")
print(f"\n  Removidos {len(pendentes)} pendentes: {pendentes}")
print(f"  Árvore reduzida: {T_reduzido.number_of_nodes()} vértices")
print(f"  Centro reduzido: {centro_reduzido}")
print(f"\n  Centros são os mesmos? {set(centro_original) == set(centro_reduzido)} ✓")

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Original
pos_orig = nx.spring_layout(T, seed=42)
cores_orig = ['red' if v in centro_original else 
              'lightcoral' if T.degree(v) == 1 else 'lightblue' for v in T.nodes()]
nx.draw(T, pos_orig, ax=axes[0], with_labels=True, node_color=cores_orig,
        node_size=700, font_size=10, font_weight='bold', edge_color='gray', width=2)
axes[0].set_title(f'Original\nCentro: {centro_original}', fontweight='bold')
axes[0].axis('off')

# Reduzido
pos_red = {v: pos_orig[v] for v in T_reduzido.nodes()}
cores_red = ['red' if v in centro_reduzido else 'lightblue' for v in T_reduzido.nodes()]
nx.draw(T_reduzido, pos_red, ax=axes[1], with_labels=True, node_color=cores_red,
        node_size=700, font_size=10, font_weight='bold', edge_color='gray', width=2)
axes[1].set_title(f'Após Remover Pendentes\nCentro: {centro_reduzido}', fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print("\n✓ O centro se PRESERVA ao remover vértices pendentes!")

## 🎯 Aplicação: Escolhendo Líder em Rede

In [ ]:
# Simular rede de comunicação
rede = nx.Graph()
rede.add_edges_from([
    ('Alice', 'Bob'), ('Bob', 'Carol'), ('Carol', 'David'),
    ('Bob', 'Eve'), ('Carol', 'Frank'),
    ('Eve', 'Grace'), ('Frank', 'Henry'),
    ('David', 'Ivy')
])

# Encontrar líder (centro)
centro_rede = nx.center(rede)
exc_rede = nx.eccentricity(rede)

# Visualizar
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(rede, seed=42)

# Cores por excentricidade
cores = ['red' if v in centro_rede else 'lightblue' for v in rede.nodes()]
tamanhos = [2000 if v in centro_rede else 1200 for v in rede.nodes()]

nx.draw_networkx_nodes(rede, pos, node_color=cores, node_size=tamanhos)
nx.draw_networkx_labels(rede, pos, font_size=11, font_weight='bold')
nx.draw_networkx_edges(rede, pos, edge_color='gray', width=2)

# Adicionar excentricidades
labels_exc = {v: f"E={exc_rede[v]}" for v in rede.nodes()}
pos_labels = {v: (x, y-0.12) for v, (x, y) in pos.items()}
nx.draw_networkx_labels(rede, pos_labels, labels_exc, font_size=9,
                       bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

plt.title(f'Rede de Comunicação\nLíder(es) Ideal(is): {centro_rede} (vermelho, maior)',
          fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"\n👥 Escolha do Líder:")
print("="*60)
print(f"  Líder(es): {centro_rede}")
print(f"  Distância máxima até qualquer pessoa: {exc_rede[centro_rede[0]]}")
print(f"\n  Por quê {centro_rede[0]} é o melhor?")
print(f"    - Excentricidade mínima ({exc_rede[centro_rede[0]]})")
print(f"    - Nenhuma pessoa está a mais de {exc_rede[centro_rede[0]]} passos")
print(f"    - Comunicação mais eficiente!")

## 🎯 Resumo

### Definições Principais

| Conceito | Definição |
|----------|----------|
| **Excentricidade E(v)** | $\max_{w \in V} d(v, w)$ |
| **Raio r(G)** | $\min_{v \in V} E(v)$ (mínima excentricidade) |
| **Diâmetro d(G)** | $\max_{v \in V} E(v)$ (máxima excentricidade) |
| **Centro C(G)** | $\{v : E(v) = r(G)\}$ |
| **Periferia P(G)** | $\{v : E(v) = d(G)\}$ |

### Propriedades de Árvores

1. **Proposição**: O centro de uma árvore tem **1 ou 2 vértices**
   - 1 vértice: **centro**
   - 2 vértices: **bicentro** (adjacentes)

2. **Proposição**: Remover vértices pendentes **preserva o centro**

### Algoritmo: Método de Eliminação Progressiva

```
ENQUANTO n > 2:
    Remover todos os vértices pendentes (grau 1)
FIM
Vértices restantes = CENTRO
```

### Complexidade

- **Eliminação**: $O(n)$ (cada vértice removido uma vez)
- **Cálculo direto**: $O(n^2)$ (calcular todas as distâncias)

### Aplicações

- 👥 Escolha de líder em redes
- 🏭 Localização de instalações
- 🌐 Posicionamento de servidores
- 🚑 Localização de centros de emergência

## 🎓 Conclusão do Módulo Árvores

Nesta série de notebooks sobre árvores, estudamos:

### 📚 Notebook 1: Introdução
- Definição de árvore (conexo + sem ciclos)
- Árvores orientadas
- Distância entre vértices
- Aplicações práticas

### 📐 Notebook 2: Propriedades
- Teorema fundamental (caminho único)
- 5 definições equivalentes
- Propriedade $n-1$ arestas
- Vértices pendentes

### 🌲 Notebook 3: Enraizadas e Binárias
- Árvores enraizadas vs. livres
- Níveis e altura
- Árvores binárias completas
- Proposições sobre vértices pendentes

### 🎯 Notebook 4: Centro (atual)
- Excentricidade, raio, diâmetro
- Centro e periferia
- Método de eliminação progressiva
- Aplicações em redes

### 🚀 Próximos Passos

Possíveis tópicos avançados:
- Algoritmos de busca (DFS, BFS)
- Árvores geradoras mínimas (Kruskal, Prim)
- Árvores de Steiner
- Árvores de decisão

## 📝 Exercícios

1. Prove que em uma árvore, $r(T) \leq d(T) \leq 2r(T)$.
2. Mostre que um caminho de $n$ vértices tem centro único se $n$ é ímpar.
3. Implemente o método de eliminação progressiva de forma iterativa.
4. Encontre uma árvore onde o centro não é o vértice de maior grau.
5. Prove que se |C(T)| = 2, então os dois vértices são adjacentes.
6. Crie uma árvore com exatamente 3 vértices na periferia.
7. Calcule o centro de uma árvore genealógica de sua escolha.